# Model Candidate Comparison

Comparing five classifiers for AHU fault detection on the same train/val
split, using per-class precision, recall, and F1 (not plain accuracy, given
the severe class imbalance found in EDA). The goal is to pick a model with
evidence, not assumption.

**Candidates**:
- Logistic Regression - linear baseline, establishes a floor
- Random Forest - tree-based bagging ensemble
- XGBoost - gradient-boosted trees, handles missing values natively
- LightGBM - gradient-boosted trees, XGBoost's closest rival implementation
- MLP (feed-forward neural net) - tests whether deep learning helps on this
  tabular dataset at this scale, where research generally favors tree-based
  methods

**Excluded from this split**: Valve position fault (117 rows, split=NULL in
the database, too thin to evaluate reliably per the EDA findings).

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("../.env")

user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

train_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'train'", engine)
val_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'val'", engine)

print(f"Train: {train_df.shape}")
print(f"Val: {val_df.shape}")
train_df["labeling"].value_counts()

Train: (105249, 23)
Val: (35083, 23)


labeling
Normal condition                71642
Supply fan fault                28789
Return air temperature fault     4199
Supply air temperature fault      619
Name: count, dtype: int64

## Feature preparation

Separating the 18 sensor columns as features from metadata columns
(id, ahu_id, reading_time, split) and the target (labeling). The target is
label-encoded to integers. XGBoost and LightGBM receive true NaN for the
structural nulls found in EDA, since both handle missing values natively.
Logistic Regression, Random Forest, and MLP cannot handle NaN, so they
receive a copy with nulls filled as -1, a sentinel value distinguishable
from any real sensor reading, rather than an imputed average that would
misrepresent an inactive heating or cooling system as active.

In [2]:
from sklearn.preprocessing import LabelEncoder

feature_cols = [c for c in train_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

le = LabelEncoder()
y_train = le.fit_transform(train_df["labeling"])
y_val = le.transform(val_df["labeling"])

print("Features:", feature_cols)
print("Classes:", list(le.classes_))

# Sentinel-filled version for models that can't handle NaN natively
X_train_filled = X_train.fillna(-1)
X_val_filled = X_val.fillna(-1)

Features: ['set_point_temp', 'return_temp', 'supply_air_temp', 'supply_fan', 'valve_position', 'heating_supply_temp_1', 'heating_supply_temp_2', 'total_heating_pump', 'heating_pump_1', 'heating_pump_2', 'heating_pump_3', 'cooling_supply_temp_1', 'cooling_supply_temp_2', 'total_cooling_pump', 'cooling_pump_1', 'cooling_pump_2', 'cooling_pump_3', 'cooling_pump_4']
Classes: ['Normal condition', 'Return air temperature fault', 'Supply air temperature fault', 'Supply fan fault']


## Training all five candidates

Each model is trained on the same training set and evaluated on the same
validation set. Logistic Regression additionally needs feature scaling
(it is sensitive to feature magnitude, unlike tree-based models), so a
scaled copy is prepared for it and MLP, which also benefits from scaled
input.

In [3]:
import time
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb

# Logistic Regression and MLP benefit from scaled features;
# tree-based models (RF, XGBoost, LightGBM) do not need this.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_val_scaled = scaler.transform(X_val_filled)

results = {}


def train_and_time(name, fit_fn):
    print(f"Training {name}...")
    start = time.time()
    model = fit_fn()
    elapsed = time.time() - start
    print(f"  done in {elapsed:.1f}s")
    return model


# 1. Logistic Regression
lr = train_and_time("Logistic Regression",
    lambda: LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train))
results["Logistic Regression"] = lr.predict(X_val_scaled)

# 2. Random Forest
rf = train_and_time("Random Forest",
    lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train_filled, y_train))
results["Random Forest"] = rf.predict(X_val_filled)

# 3. XGBoost - receives true NaN, not the filled version
xgb_model = train_and_time("XGBoost",
    lambda: xgb.XGBClassifier(random_state=42, eval_metric="mlogloss").fit(X_train, y_train))
results["XGBoost"] = xgb_model.predict(X_val)

# 4. LightGBM - also handles true NaN natively
lgb_model = train_and_time("LightGBM",
    lambda: lgb.LGBMClassifier(random_state=42, verbose=-1).fit(X_train, y_train))
results["LightGBM"] = lgb_model.predict(X_val)

# 5. MLP - scaled input, same as Logistic Regression
mlp = train_and_time("MLP",
    lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42).fit(X_train_scaled, y_train))
results["MLP"] = mlp.predict(X_val_scaled)

print("\nAll five models trained")

Training Logistic Regression...
  done in 8.2s
Training Random Forest...
  done in 2.3s
Training XGBoost...
  done in 4.1s
Training LightGBM...
  done in 1.2s
Training MLP...
  done in 23.1s

All five models trained


## Comparing metrics

Printing full per-class precision, recall, and F1 for each model,
macro F1 as the primary summary number given our class imbalance.

In [4]:
from sklearn.metrics import f1_score

summary_rows = []

for name, preds in results.items():
    report = classification_report(y_val, preds, target_names=le.classes_, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]
    weighted_f1 = report["weighted avg"]["f1-score"]
    summary_rows.append({"model": name, "macro_f1": macro_f1, "weighted_f1": weighted_f1})

    print(f"=== {name} ===")
    print(classification_report(y_val, preds, target_names=le.classes_))
    print()

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
summary_df

=== Logistic Regression ===
                              precision    recall  f1-score   support

            Normal condition       0.88      0.86      0.87     23880
Return air temperature fault       0.63      0.46      0.53      1400
Supply air temperature fault       0.62      0.02      0.05       206
            Supply fan fault       0.72      0.80      0.76      9597

                    accuracy                           0.82     35083
                   macro avg       0.71      0.54      0.55     35083
                weighted avg       0.82      0.82      0.82     35083


=== Random Forest ===
                              precision    recall  f1-score   support

            Normal condition       1.00      1.00      1.00     23880
Return air temperature fault       0.99      0.98      0.98      1400
Supply air temperature fault       0.95      0.91      0.93       206
            Supply fan fault       1.00      1.00      1.00      9597

                    accuracy      

,model,macro_f1,weighted_f1
3,LightGBM,0.987667,0.998572
2,XGBoost,0.986220,0.998289
4,MLP,0.982158,0.997016
1,Random Forest,0.977568,0.997851
0,Logistic Regression,0.552156,0.821892


In [5]:
# Check 1: Do train and val share any rows (by id)?
overlap = set(train_df["id"]) & set(val_df["id"])
print(f"Overlapping row ids between train and val: {len(overlap)}")

# Check 2: Does any single feature perfectly separate the classes?
# Group by label and look at each feature's range - if a feature never
# overlaps between classes, it alone could explain perfect scores.
for col in feature_cols:
    ranges = train_df.groupby("labeling")[col].agg(["min", "max"])
    print(f"\n{col}")
    print(ranges)

Overlapping row ids between train and val: 0

set_point_temp
                               min   max
labeling                                
Normal condition              10.0  30.0
Return air temperature fault  18.0  40.0
Supply air temperature fault  10.0  34.0
Supply fan fault              18.0  32.0

return_temp
                               min    max
labeling                                 
Normal condition              5.82  29.57
Return air temperature fault  3.67  31.71
Supply air temperature fault  5.81  29.15
Supply fan fault             -1.22  32.89

supply_air_temp
                                min    max
labeling                                  
Normal condition               9.70  48.76
Return air temperature fault  10.32  48.90
Supply air temperature fault  11.18  30.61
Supply fan fault               8.52  45.95

supply_fan
                              min  max
labeling                              
Normal condition              0.0  1.0
Return air temperature f

In [6]:
# Check 3: XGBoost feature importance - is it spread out or dominated
# by one feature?
importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance

total_heating_pump       0.487243
supply_fan               0.188061
total_cooling_pump       0.127297
return_temp              0.076095
set_point_temp           0.055596
heating_pump_1           0.022400
valve_position           0.022299
supply_air_temp          0.008065
heating_pump_3           0.002938
heating_supply_temp_1    0.002803
heating_pump_2           0.002445
heating_supply_temp_2    0.001125
cooling_pump_1           0.000955
cooling_supply_temp_2    0.000921
cooling_supply_temp_1    0.000910
cooling_pump_2           0.000527
cooling_pump_3           0.000321
cooling_pump_4           0.000000
dtype: float32

## Conclusion

| Model | Macro F1 | Weighted F1 | Train time |
|---|---|---|---|
| LightGBM | 0.9877 | 0.9986 | 1.2s |
| XGBoost | 0.9862 | 0.9983 | 4.1s |
| MLP | 0.9822 | 0.9970 | 23.1s |
| Random Forest | 0.9776 | 0.9979 | 2.3s |
| Logistic Regression | 0.5522 | 0.8219 | 8.2s |

**Decision: XGBoost.**

LightGBM scored marginally higher and trained faster, but the two are close
enough (within half a point of macro F1) that either is defensible. XGBoost
was chosen specifically because SHAP's tree explainer has the most mature
support for it, which matters directly for the next step (per-instance
explainability), and because it is the more widely recognized tool for this
kind of task.

Logistic Regression's near-total failure on Supply air temperature fault
(recall 0.02) confirms this problem needs non-linear modeling. MLP came
close to the tree-based models but took roughly 20x longer to train for no
accuracy gain, consistent with established findings that deep learning
rarely beats gradient boosting on tabular data at this scale.

**On the near-perfect scores**: verified this is not data leakage (zero
overlapping row ids between train and val) and not a single feature trivially
separating the classes (every feature's value range overlaps across all four
labels). XGBoost's feature importance is spread across several sensors
(total_heating_pump, supply_fan, total_cooling_pump, return_temp, and
set_point_temp together account for most of the model's decisions), not
dominated by one column. The high accuracy reflects that these fault labels
were originally defined using explicit threshold rules on these same
sensors (Table 7 of the source paper), so a model capable of learning
combined threshold logic is expected to perform very well here. This is
disclosed plainly rather than presented as an unusually strong result.

**Not addressed in this notebook**: Valve position fault (excluded, too few
samples to evaluate reliably) and hyperparameter tuning (XGBoost was run
with default settings here; tuning happens in the next step before the
final frozen model is produced).